# 03 - AF3 UPO preparation with Compound-I waters and Rosetta relax

## Introduction

This tutorial continues the AF3 workflow from tutorial 02.

Starting point:
- AF3 models already contain protein + heme + Mg.

Goal:
- Add missing Compound-I ferryl oxygen and catalytic waters.
- Apply residue normalization for Rosetta (`CYX` and conditional `HIP`).
- Run Rosetta relax with active-site constraints.
- Export one relaxed model per UPO.

This notebook uses input files bundled in this tutorial folder (with fallback to `example_code/`).


## 1. Import modules

We keep the same style as previous tutorials and include `bsc_calculations` for SLURM arrays.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

import prepare_proteins
import bsc_calculations

from Bio import PDB
from Bio.PDB import Superimposer


## 2. Resolve project paths and inputs

This notebook first looks for local inputs inside this tutorial folder, and falls back to `example_code/` when needed.


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_proteins").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing 'prepare_proteins'.")

def resolve_input_base(repo_root: Path) -> Path:
    local_base = repo_root / "docs" / "tutorial" / "03-AF3UPOCompoundIWatersRosetta"
    local_ok = (
        (local_base / "cofactor" / "cofactor_A.pdb").exists()
        and (local_base / "params" / "HUP.params").exists()
        and (local_base / "pdbs_nocofactor" / "CviUPO_Chaetomium.pdb").exists()
    )
    if local_ok:
        return local_base

    fallback = repo_root / "example_code"
    fallback_ok = (
        (fallback / "cofactor" / "cofactor_A.pdb").exists()
        and (fallback / "params" / "HUP.params").exists()
        and (fallback / "pdbs_nocofactor" / "CviUPO_Chaetomium.pdb").exists()
    )
    if fallback_ok:
        return fallback

    raise FileNotFoundError(
        "Could not find required inputs in tutorial folder or example_code."
    )

repo_root = find_repo_root(Path.cwd().resolve())
input_base = resolve_input_base(repo_root)

cofactor_dir = input_base / "cofactor"
params_dir = input_base / "params"
reference_pdb = input_base / "pdbs_nocofactor" / "CviUPO_Chaetomium.pdb"

af3_candidates = [
    repo_root / "af3_upo_heme_mg_filtered_models",
    repo_root / "af3_upo_heme_mg_top_models",
]
af3_models_folder = next((p for p in af3_candidates if p.exists()), None)
if af3_models_folder is None:
    raise FileNotFoundError(
        "AF3 models folder not found. Expected one of: "
        + ", ".join(str(p) for p in af3_candidates)
    )

print("repo_root:", repo_root)
print("input_base:", input_base)
print("AF3 input:", af3_models_folder)
print("reference_pdb:", reference_pdb)
print("cofactor template:", cofactor_dir / "cofactor_A.pdb")
print("params_dir:", params_dir)


## 3. Load AF3-selected models


In [ ]:
models = prepare_proteins.proteinModels(str(af3_models_folder))
print("Loaded models:", len(models.models_names))
print(models.models_names[:10])


## 4. Basic cofactor QC

We keep models with at least one heme-like residue (`HEM`/`HUP`) and one `MG` residue.


In [ ]:
def has_required_cofactors(structure):
    has_heme = False
    has_mg = False
    for residue in structure.get_residues():
        if residue.resname in {"HEM", "HUP"}:
            has_heme = True
        if residue.resname == "MG":
            has_mg = True
    return has_heme and has_mg

remove_models = []
for model_name in models.models_names:
    structure = models.structures[model_name]
    if not has_required_cofactors(structure):
        remove_models.append(model_name)

for model_name in remove_models:
    models.removeModel(model_name)

print(f"Removed {len(remove_models)} models missing HEM/HUP or MG")
print(f"Remaining models: {len(models.models_names)}")


## 5. Align models to reference (single orientation)

Unlike the previous prototype notebook, we do **not** branch into multiple cofactor orientations.
We keep one orientation and only transfer missing Compound-I oxygen and catalytic waters.


In [ ]:
aligned_folder = "03_aligned_models"
models.alignModelsToReferencePDB(str(reference_pdb), aligned_folder, chain_indexes=0)
prep_models = prepare_proteins.proteinModels(aligned_folder)
print("Aligned models loaded:", len(prep_models.models_names))


## 6. Add missing ferryl oxygen and catalytic waters from template

We use `cofactor/cofactor_A.pdb` as single-orientation template.

Strategy per model:
- Superimpose template heme (`HUP`) on model heme (`HEM`/`HUP`) using shared heme atoms.
- Add ferryl oxygen (`O`) if missing.
- Add template waters only when no equivalent water already exists nearby.
- Rename heme residue to `HUP` for Rosetta params compatibility.


In [ ]:
cofactor_template = PDB.PDBParser(QUIET=True).get_structure(
    "cof", str(cofactor_dir / "cofactor_A.pdb")
)

# Template residues
template_hup = None
template_waters = []
for residue in cofactor_template.get_residues():
    if residue.resname == "HUP":
        template_hup = residue
    elif residue.resname in {"HOH", "WAT"} and "O" in residue:
        template_waters.append(residue)

if template_hup is None:
    raise ValueError("Template HUP residue not found in cofactor_A.pdb")

anchor_priority = [
    "FE", "NA", "NB", "NC", "ND", "CHA", "CHB", "CHC", "CHD",
    "C1A", "C1B", "C1C", "C1D", "C2A", "C2B", "C2C", "C2D",
]

def find_heme_residue(structure):
    for residue in structure.get_residues():
        if residue.resname in {"HEM", "HUP"} and "FE" in residue:
            return residue
    return None

def find_mg_residue(structure):
    for residue in structure.get_residues():
        if residue.resname == "MG" and "MG" in residue:
            return residue
    return None

def transform_coord(coord, rot, tran):
    return np.dot(coord, rot) + tran

def collect_water_oxygen_coords(structure):
    coords = []
    for residue in structure.get_residues():
        if residue.resname in {"HOH", "WAT"} and "O" in residue:
            coords.append(residue["O"].coord)
    if not coords:
        return np.empty((0, 3), dtype=float)
    return np.array(coords, dtype=float)

failed_models = []
for model_name in prep_models.models_names:
    structure = prep_models.structures[model_name]
    heme_res = find_heme_residue(structure)
    mg_res = find_mg_residue(structure)

    if heme_res is None or mg_res is None:
        failed_models.append(model_name)
        continue

    model_atom_names = {a.name for a in heme_res.get_atoms()}
    template_atom_names = {a.name for a in template_hup.get_atoms()}
    anchor_names = [a for a in anchor_priority if a in model_atom_names and a in template_atom_names]

    if len(anchor_names) < 6:
        failed_models.append(model_name)
        continue

    fixed_atoms = [heme_res[a] for a in anchor_names]
    moving_atoms = [template_hup[a] for a in anchor_names]
    sup = Superimposer()
    sup.set_atoms(fixed_atoms, moving_atoms)
    rot, tran = sup.rotran

    # Add ferryl oxygen if missing
    if "O" not in model_atom_names and "O" in template_hup:
        o_coord = transform_coord(template_hup["O"].coord, rot, tran)
        serial = max(atom.serial_number for atom in structure.get_atoms()) + 1
        oxygen_atom = PDB.Atom.Atom(
            "O",
            np.array(o_coord, dtype=float),
            0.0,
            1.0,
            " ",
            " O  ",
            serial,
            "O",
        )
        heme_res.add(oxygen_atom)

    # Add catalytic waters only if absent nearby
    existing_water_coords = collect_water_oxygen_coords(structure)
    mg_chain = mg_res.get_parent().id

    for water_res in template_waters:
        w_coord = transform_coord(water_res["O"].coord, rot, tran)
        if existing_water_coords.shape[0] > 0:
            dmin = np.linalg.norm(existing_water_coords - w_coord, axis=1).min()
            if dmin < 0.8:
                continue

        prep_models.addResidueToModel(
            model=model_name,
            chain_id=mg_chain,
            resname="HOH",
            atom_names=["O"],
            coordinates=np.array([w_coord], dtype=float),
            water=True,
        )

        if existing_water_coords.shape[0] == 0:
            existing_water_coords = np.array([w_coord], dtype=float)
        else:
            existing_water_coords = np.vstack([existing_water_coords, w_coord])

    # Force HUP naming for Rosetta params
    heme_res.resname = "HUP"

for model_name in failed_models:
    prep_models.removeModel(model_name)

print(f"Removed {len(failed_models)} models during O/water placement")
print(f"Remaining models: {len(prep_models.models_names)}")


In [ ]:
# Quick sanity report
for model_name in prep_models.models_names[:10]:
    structure = prep_models.structures[model_name]
    heme = None
    mg = None
    n_waters = 0
    for residue in structure.get_residues():
        if residue.resname == "HUP" and "FE" in residue:
            heme = residue
        if residue.resname == "MG":
            mg = residue
        if residue.resname in {"HOH", "WAT"}:
            n_waters += 1

    has_oxo = heme is not None and "O" in heme
    print(model_name, "HUP:", heme is not None, "Oxo:", has_oxo, "MG:", mg is not None, "waters:", n_waters)


## 7. Save intermediate prepared models


In [ ]:
prepared_stage1_folder = "03_prepared_with_oxo_waters"
prep_models.saveModels(prepared_stage1_folder)
print("Saved:", prepared_stage1_folder)


## 8. Build MSA and map catalytic positions from CviUPO reference

Reference residue positions (from previous protocol):
- His: 90
- Cys: 19
- Glu/Asp (saline): 162
- Arg alternative: 155
- Mg-Glu/Asp/Gln: 89
- Mg-Ser: 93


In [ ]:
ref_models = prepare_proteins.proteinModels(str(reference_pdb.parent), only_models=["CviUPO_Chaetomium"])
reference_name = "CviUPO_Chaetomium"
reference_seq = ref_models.sequences[reference_name]

msa = prep_models.calculateMSA(extra_sequences={reference_name: reference_seq})

def map_reference_position(pos):
    idx = prepare_proteins.alignment.msaIndexesFromSequencePositions(msa, reference_name, [pos])
    return prep_models.getStructurePositionsFromMSAindexes(idx)

histidines = map_reference_position(90)
cysteines = map_reference_position(19)
glutamic = map_reference_position(162)
arginines = map_reference_position(155)
mg_glutamic = map_reference_position(89)
mg_serine = map_reference_position(93)

print("Position mapping completed")


## 9. Filter incompatible models and classify catalytic residue types


In [ ]:
def get_polymer_residue_by_resid(structure, resid):
    if resid is None:
        return None
    for residue in structure.get_residues():
        if residue.id[0] == " " and residue.id[1] == resid:
            return residue
    return None

his_type = {}
glu_type = {}
mg_glutamic_type = {}
invalid_models = set()

for model_name in prep_models.models_names:
    structure = prep_models.structures[model_name]

    his_pos = histidines[model_name][0] if histidines[model_name] else None
    cys_pos = cysteines[model_name][0] if cysteines[model_name] else None
    glu_pos = glutamic[model_name][0] if glutamic[model_name] else None
    arg_pos = arginines[model_name][0] if arginines[model_name] else None
    mg_glu_pos = mg_glutamic[model_name][0] if mg_glutamic[model_name] else None
    mg_ser_pos = mg_serine[model_name][0] if mg_serine[model_name] else None

    his_res = get_polymer_residue_by_resid(structure, his_pos)
    cys_res = get_polymer_residue_by_resid(structure, cys_pos)
    glu_res = get_polymer_residue_by_resid(structure, glu_pos)
    arg_res = get_polymer_residue_by_resid(structure, arg_pos)
    mg_glu_res = get_polymer_residue_by_resid(structure, mg_glu_pos)
    mg_ser_res = get_polymer_residue_by_resid(structure, mg_ser_pos)

    if his_res is not None and his_res.resname in {"HIS", "HIP", "HID", "HIE"}:
        his_type[model_name] = "HIS"
    elif arg_res is not None and arg_res.resname == "ARG":
        his_type[model_name] = "ARG"
    else:
        invalid_models.add(model_name)

    if cys_res is None or cys_res.resname not in {"CYS", "CYX"}:
        invalid_models.add(model_name)

    if glu_res is None or glu_res.resname not in {"GLU", "ASP"}:
        invalid_models.add(model_name)
    else:
        glu_type[model_name] = glu_res.resname

    if mg_glu_res is None or mg_glu_res.resname not in {"GLU", "ASP", "GLN"}:
        invalid_models.add(model_name)
    else:
        mg_glutamic_type[model_name] = mg_glu_res.resname

    if mg_ser_res is None or mg_ser_res.resname != "SER":
        invalid_models.add(model_name)

for model_name in sorted(invalid_models):
    prep_models.removeModel(model_name)

print(f"Removed {len(invalid_models)} incompatible models")
print(f"Remaining models: {len(prep_models.models_names)}")


## 10. Normalize residue names for Rosetta (`HIP` and `CYX`)

Rules:
- Histidine saline variant -> `HIP` when the model uses histidine at the target position.
- Catalytic cysteine -> `CYX` and remove `HG` if present.


In [ ]:
def apply_rosetta_residue_naming(models_obj):
    for model_name in models_obj.models_names:
        structure = models_obj.structures[model_name]

        his_pos = histidines[model_name][0] if histidines[model_name] else None
        cys_pos = cysteines[model_name][0] if cysteines[model_name] else None

        for residue in structure.get_residues():
            if residue.id[0] != " ":
                continue

            if his_pos is not None and residue.id[1] == his_pos and his_type.get(model_name) == "HIS":
                residue.resname = "HIP"

            if cys_pos is not None and residue.id[1] == cys_pos:
                residue.resname = "CYX"
                if "HG" in residue:
                    residue.detach_child("HG")

apply_rosetta_residue_naming(prep_models)
print("Applied HIP/CYX normalization")


## 11. Build active-site constraint files (single orientation)

Constraints include:
- `CYX SG` - `HUP FE`
- `HUP FE` - ferryl `O`
- saline bond (`HIP NE2` or `ARG CZ` to acidic residue)
- Mg coordination (protein + cofactor)
- catalytic water interactions


In [ ]:
cst_root = Path("03_cst_files")
cst_root.mkdir(exist_ok=True)

def residue_by_resid(structure, resid):
    for residue in structure.get_residues():
        if residue.id[0] == " " and residue.id[1] == resid:
            return residue
    return None

def find_residue(structure, resname):
    for residue in structure.get_residues():
        if residue.resname == resname:
            return residue
    return None

def residue_atom_label(residue, atom_name):
    return f"{atom_name} {residue.id[1]}{residue.get_parent().id}"

cst_files = {}

for model_name in prep_models.models_names:
    structure = prep_models.structures[model_name]

    hup = find_residue(structure, "HUP")
    mg = find_residue(structure, "MG")

    cys_res = residue_by_resid(structure, cysteines[model_name][0])
    his_res = residue_by_resid(structure, histidines[model_name][0]) if histidines[model_name][0] is not None else None
    arg_res = residue_by_resid(structure, arginines[model_name][0]) if arginines[model_name][0] is not None else None
    glu_res = residue_by_resid(structure, glutamic[model_name][0])
    mg_glu_res = residue_by_resid(structure, mg_glutamic[model_name][0])
    mg_ser_res = residue_by_resid(structure, mg_serine[model_name][0])

    # Pick saline water as closest HOH/WAT oxygen to glutamic sidechain oxygens
    candidate_waters = [
        r for r in structure.get_residues()
        if r.resname in {"HOH", "WAT"} and "O" in r
    ]

    glut_oxy_names = ["OE1", "OE2"] if glu_res.resname == "GLU" else ["OD1", "OD2"]
    glut_oxy_atoms = [glu_res[a] for a in glut_oxy_names if a in glu_res]

    saline_wat = None
    if glut_oxy_atoms and candidate_waters:
        best_d = np.inf
        for wat in candidate_waters:
            d = min(np.linalg.norm(wat["O"].coord - ga.coord) for ga in glut_oxy_atoms)
            if d < best_d:
                best_d = d
                saline_wat = wat

    required_hup_atoms = ["FE", "O", "O1A", "O2A", "O1D", "O2D"]
    missing_hup_atoms = [a for a in required_hup_atoms if a not in hup]
    if missing_hup_atoms:
        raise ValueError(f"Model {model_name} is missing required HUP atoms for constraints: {missing_hup_atoms}")

    cst_dir = cst_root / model_name
    cst_dir.mkdir(exist_ok=True)
    cst_path = cst_dir / f"{model_name}.cst"

    with open(cst_path, "w") as cf:
        # CYX SG - HUP FE
        cf.write(
            f"AtomPair {residue_atom_label(cys_res, 'SG')} {residue_atom_label(hup, 'FE')} HARMONIC 2.3 0.1\n"
        )

        # Ferryl O - FE
        cf.write(
            f"AtomPair {residue_atom_label(hup, 'O')} {residue_atom_label(hup, 'FE')} HARMONIC 1.65 0.05\n"
        )

        # Saline bond
        acidic_carbon = "CD" if glu_res.resname == "GLU" else "CG"
        if his_type[model_name] == "HIS" and his_res is not None:
            cf.write(
                f"AtomPair {residue_atom_label(his_res, 'NE2')} {residue_atom_label(glu_res, acidic_carbon)} FLAT_HARMONIC 5.0 0.3 0.2\n"
            )
        elif his_type[model_name] == "ARG" and arg_res is not None:
            cf.write(
                f"AtomPair {residue_atom_label(arg_res, 'CZ')} {residue_atom_label(glu_res, acidic_carbon)} FLAT_HARMONIC 5.0 0.3 0.2\n"
            )

        # MG - acidic residue near Mg
        if mg_glutamic_type[model_name] == "GLU":
            oxy = ["OE1", "OE2"]
        elif mg_glutamic_type[model_name] == "ASP":
            oxy = ["OD1", "OD2"]
        else:
            oxy = ["OE1", "NE2"]

        cf.write("AmbiguousConstraint\n")
        for atom_name in oxy:
            cf.write(
                f"AtomPair {residue_atom_label(mg, 'MG')} {residue_atom_label(mg_glu_res, atom_name)} HARMONIC 2.1 0.1\n"
            )
        cf.write("END\n")

        # MG - serine
        cf.write(
            f"AtomPair {residue_atom_label(mg, 'MG')} {residue_atom_label(mg_ser_res, 'OG')} HARMONIC 2.1 0.1\n"
        )

        # Cofactor COO- + serine + MG (single orientation)
        cf.write("AmbiguousConstraint\n")
        cf.write("MultiConstraint\n")
        cf.write(
            f"AtomPair {residue_atom_label(mg, 'MG')} {residue_atom_label(hup, 'O1A')} HARMONIC 2.1 0.1\n"
        )
        cf.write(
            f"AtomPair {residue_atom_label(hup, 'O2A')} {residue_atom_label(mg_ser_res, 'OG')} HARMONIC 2.7 0.5\n"
        )
        cf.write("END\n")
        cf.write("MultiConstraint\n")
        cf.write(
            f"AtomPair {residue_atom_label(mg, 'MG')} {residue_atom_label(hup, 'O2A')} HARMONIC 2.1 0.1\n"
        )
        cf.write(
            f"AtomPair {residue_atom_label(hup, 'O1A')} {residue_atom_label(mg_ser_res, 'OG')} HARMONIC 2.7 0.5\n"
        )
        cf.write("END\n")
        cf.write("END\n")

        if saline_wat is not None:
            # Water - glutamic/aspartic
            cf.write("AmbiguousConstraint\n")
            for atom_name in glut_oxy_names:
                if atom_name in glu_res:
                    cf.write(
                        f"AtomPair {residue_atom_label(saline_wat, 'O')} {residue_atom_label(glu_res, atom_name)} HARMONIC 2.7 0.5\n"
                    )
            cf.write("END\n")

            # Water - cofactor acidic oxygens
            cf.write("AmbiguousConstraint\n")
            for atom_name in ["O1D", "O2D"]:
                cf.write(
                    f"AtomPair {residue_atom_label(saline_wat, 'O')} {residue_atom_label(hup, atom_name)} HARMONIC 2.7 0.5\n"
                )
            cf.write("END\n")

    cst_files[model_name] = str(cst_path)

print(f"Generated {len(cst_files)} constraint files in {cst_root}")


## 12. Set up Rosetta relax jobs

We use provided non-standard Rosetta params:
- `HUP.params`
- `CYX.params`
- `HIP.params`


In [ ]:
params_files = sorted(str(p) for p in params_dir.glob("*.params"))
print("Params:", params_files)

relax_folder = "03_rosetta_relax"
jobs = prep_models.setUpRosettaOptimization(
    relax_folder=relax_folder,
    param_files=params_files,
    nstruct=100,
    relax_cycles=5,
    cst_files=cst_files,
    cst_optimization=False,
    skip_finished=True,
    parallelisation="srun",
)

print(f"Prepared {len(jobs)} Rosetta jobs")
if jobs:
    print("\nExample job command:\n")
    print(jobs[0])


## 13. Create SLURM array script with `bsc_calculations` (MN5)


In [ ]:
slurm_script = "slurm_array_upo_compoundI_relax.sh"
slurm_job_name = "UPO_CMPD1_RELAX"

bsc_calculations.mn5.jobArrays(
    jobs,
    script_name=slurm_script,
    job_name=slurm_job_name,
    output=slurm_job_name,
    partition="gp_bscls",
    program="rosetta",
    ntasks=1,
    cpus_per_task=1,
)

print(f"Generated {slurm_script}")
print(f"Submit with: sbatch {slurm_script}")

launch_slurm = False
if launch_slurm:
    subprocess.run(["sbatch", slurm_script], check=True)
else:
    print("Dry run only. Set launch_slurm=True to submit.")


## 14. Load best relaxed pose per model and save final prepared structures

After Rosetta jobs finish, this reads each `<model>_relax.out`, extracts the best score pose, and saves one final PDB per model.


In [ ]:
prep_models.loadModelsFromRosettaOptimization(
    relax_folder,
    filter_score_term="score",
    min_value=True,
    wat_to_hoh=False,
)

final_models_folder = "03_final_models"
prep_models.saveModels(final_models_folder)
print("Saved final models to:", final_models_folder)


## 15. Optional geometry sanity checks


In [ ]:
def first_residue_by_name(structure, name):
    for residue in structure.get_residues():
        if residue.resname == name:
            return residue
    return None

rows = []
for model_name in prep_models.models_names:
    structure = prep_models.structures[model_name]
    hup = first_residue_by_name(structure, "HUP")
    mg = first_residue_by_name(structure, "MG")
    ser = residue_by_resid(structure, mg_serine[model_name][0])

    fe_o = np.nan
    mg_ser = np.nan

    if hup is not None and "FE" in hup and "O" in hup:
        fe_o = np.linalg.norm(hup["FE"].coord - hup["O"].coord)

    if mg is not None and "MG" in mg and ser is not None and "OG" in ser:
        mg_ser = np.linalg.norm(mg["MG"].coord - ser["OG"].coord)

    rows.append({
        "model": model_name,
        "fe_o": fe_o,
        "mg_ser": mg_ser,
    })

summary_df = pd.DataFrame(rows).set_index("model").sort_index()
summary_df.head()


## 16. Notes

- This workflow assumes one cofactor orientation only.
- It is designed for AF3 outputs already containing heme and Mg.
- If your inputs already contain catalytic waters and ferryl oxygen, placement steps should mostly skip additions.
- For stricter active-site ranking, you can extend with `analyseRosettaCalculation(...)` as in the prototype notebook.
